# Vectorless RAG: Hierarchical Index over Financial Documents

**Course:** Applied NLP / LLM Systems  
**Topic:** Retrieval-Augmented Generation without Vector Search  
**Document:** BSE Corporate Filing — Annual Report (PDF)

---

## Overview

This notebook demonstrates **Vectorless RAG** — a retrieval strategy that builds a hierarchical document index and queries it through structured reasoning, without embedding vectors or approximate nearest-neighbour search.

### What You Will Learn

| Concept | Description |
|---|---|
| Hierarchical Index | Tree of document nodes, each holding a page-level summary |
| Vectorless Retrieval | Rule-based and LLM-guided traversal instead of cosine similarity |
| Traditional RAG | Baseline comparison using chunking and vector search |
| Hybrid RAG | Combining both approaches for production-grade systems |

---

## 1. Background

### 1.1 Traditional RAG

Traditional RAG pipelines follow three steps:

1. **Chunk** the document into fixed-size text segments.
2. **Embed** each chunk using a dense encoder (e.g., `text-embedding-ada-002`).
3. **Retrieve** the top-k chunks by cosine similarity to the query embedding.

This works well at scale but suffers from two known failure modes:
- Important context is split across chunk boundaries.
- The retriever cannot reason about document *structure* (sections, tables, hierarchies).

### 1.2 Vectorless RAG

Vectorless RAG replaces the embedding retriever with a **structured index traversal**:

- The document is parsed into a tree of nodes (root → sections → pages).
- Each node stores a short **summary** and optional metadata (page number, section title).
- At query time, the model reads the summaries top-down and descends into relevant subtrees.

No vectors are stored. Retrieval is driven by **language understanding** of the summaries.

### 1.3 Comparison

| Dimension | Traditional RAG | Vectorless RAG | Hybrid RAG |
|---|---|---|---|
| Storage | Vector database | In-memory tree | Both |
| Retrieval | Approximate NN | Structured traversal | Combined |
| Handles structure | Weak | Strong | Strong |
| Scalability | High | Medium | High |
| Best for | Large corpora | Structured docs | Complex systems |

---

## 2. Setup and Installation

In [ ]:
# Install required packages
!pip install -q pypdf llama-index llama-index-llms-openai openai

In [ ]:
import os
import requests
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional

from pypdf import PdfReader
from openai import OpenAI

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk-..."   # Replace with your key

client = OpenAI()
print("Setup complete.")

---

## 3. Load the Financial Report PDF

In [ ]:
PDF_URL = (
    "https://www.bseindia.com/xml-data/corpfiling/AttachHis/"
    "faa17496-838f-432d-a39f-32d82156799c.pdf"
)
PDF_PATH = Path("annual_report.pdf")

# Download only if not already present
if not PDF_PATH.exists():
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(PDF_URL, headers=headers)
    response.raise_for_status()
    PDF_PATH.write_bytes(response.content)
    print(f"Downloaded {PDF_PATH} ({PDF_PATH.stat().st_size // 1024} KB)")
else:
    print(f"Using cached file: {PDF_PATH}")

# Extract page text
reader = PdfReader(str(PDF_PATH))
pages = [page.extract_text() or "" for page in reader.pages]
print(f"Total pages extracted: {len(pages)}")

---

## 4. Building the Hierarchical Index

### 4.1 Node Definition

Each node in the index tree holds:
- The **raw text** of a page (or a group of pages at higher levels).
- A **summary** generated by the LLM — used for traversal decisions.
- **Children** links to sub-nodes (empty for leaf/page nodes).

In [ ]:
@dataclass
class DocumentNode:
    """A node in the hierarchical document index."""
    node_id: str
    level: int                        # 0 = root, 1 = section, 2 = page
    page_range: tuple                 # (start_page, end_page), 1-indexed
    raw_text: str
    summary: str = ""
    children: List["DocumentNode"] = field(default_factory=list)
    parent: Optional["DocumentNode"] = field(default=None, repr=False)

    def __repr__(self):
        return (
            f"DocumentNode(id={self.node_id!r}, level={self.level}, "
            f"pages={self.page_range}, children={len(self.children)})"
        )

### 4.2 LLM Summarisation Helper

In [ ]:
def summarise(text: str, max_tokens: int = 120) -> str:
    """Return a concise summary of `text` using the LLM."""
    prompt = (
        "Summarise the following financial document excerpt in 2-3 sentences. "
        "Be specific about numbers, section names, and key facts.\n\n"
        f"{text[:3000]}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0,
    )
    return response.choices[0].message.content.strip()

### 4.3 Index Construction

The tree is built bottom-up:
1. **Leaf nodes** — one per page, summarised individually.
2. **Section nodes** — groups of `section_size` pages; summary is derived from child summaries.
3. **Root node** — single node whose summary combines all section summaries.

In [ ]:
def build_index(pages: List[str], section_size: int = 5) -> DocumentNode:
    """
    Build a two-level hierarchical index over a list of page texts.

    Parameters
    ----------
    pages        : list of raw page strings (0-indexed internally)
    section_size : number of pages per section node

    Returns
    -------
    root : DocumentNode at level 0
    """
    print(f"Building index for {len(pages)} pages ...")

    # --- Level 2: leaf nodes (one per page) ---
    leaf_nodes = []
    for i, text in enumerate(pages):
        node = DocumentNode(
            node_id=f"page_{i+1}",
            level=2,
            page_range=(i + 1, i + 1),
            raw_text=text,
        )
        if text.strip():
            node.summary = summarise(text)
        else:
            node.summary = "[Empty page]"
        print(f"  Page {i+1}/{len(pages)} summarised.")
        leaf_nodes.append(node)

    # --- Level 1: section nodes ---
    section_nodes = []
    for idx in range(0, len(leaf_nodes), section_size):
        chunk = leaf_nodes[idx : idx + section_size]
        combined_text = "\n\n".join(n.raw_text for n in chunk)
        combined_summary = "\n".join(n.summary for n in chunk)
        sec_node = DocumentNode(
            node_id=f"section_{idx // section_size + 1}",
            level=1,
            page_range=(chunk[0].page_range[0], chunk[-1].page_range[1]),
            raw_text=combined_text,
            children=chunk,
        )
        sec_node.summary = summarise(combined_summary)
        for child in chunk:
            child.parent = sec_node
        section_nodes.append(sec_node)
        print(f"  Section {sec_node.node_id} built (pages {sec_node.page_range}).")

    # --- Level 0: root node ---
    root_summary_input = "\n".join(n.summary for n in section_nodes)
    root = DocumentNode(
        node_id="root",
        level=0,
        page_range=(1, len(pages)),
        raw_text="",
        children=section_nodes,
        summary=summarise(root_summary_input, max_tokens=200),
    )
    for child in section_nodes:
        child.parent = root

    print("Index construction complete.")
    return root


# Build the index (uses the first 20 pages to stay within demo limits)
DEMO_PAGES = pages[:20]
root = build_index(DEMO_PAGES, section_size=5)

### 4.4 Inspect the Index

In [ ]:
def print_tree(node: DocumentNode, indent: int = 0) -> None:
    prefix = "  " * indent
    print(f"{prefix}{node}")
    print(f"{prefix}  Summary: {node.summary[:100]}...")
    for child in node.children:
        print_tree(child, indent + 1)

print_tree(root)

---

## 5. Querying the Hierarchical Index

The retriever traverses the tree top-down:
1. **Select** the most relevant section node based on summaries.
2. **Select** the most relevant page node within that section.
3. **Answer** the query from the leaf node's raw text.

In [ ]:
def select_best_child(query: str, nodes: List[DocumentNode]) -> DocumentNode:
    """
    Ask the LLM which child node is most relevant to the query.
    Returns the chosen DocumentNode.
    """
    options = "\n".join(
        f"[{i}] {n.node_id} (pages {n.page_range}): {n.summary}"
        for i, n in enumerate(nodes)
    )
    prompt = (
        f"You are navigating a document index. "
        f"Choose the single most relevant node for the query below.\n\n"
        f"Query: {query}\n\n"
        f"Nodes:\n{options}\n\n"
        f"Reply with only the integer index (0, 1, 2 ...) of the best node."
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=5,
        temperature=0,
    )
    idx_str = response.choices[0].message.content.strip()
    idx = int(idx_str) if idx_str.isdigit() else 0
    idx = max(0, min(idx, len(nodes) - 1))
    return nodes[idx]


def vectorless_query(query: str, root: DocumentNode) -> str:
    """
    Traverse the hierarchical index to answer `query`.
    """
    print(f"Query: {query}")

    # Step 1: pick the best section
    section = select_best_child(query, root.children)
    print(f"  Selected section: {section.node_id} (pages {section.page_range})")

    # Step 2: pick the best page within that section
    page_node = select_best_child(query, section.children)
    print(f"  Selected page: {page_node.node_id}")

    # Step 3: answer from raw page text
    answer_prompt = (
        f"Use the document excerpt below to answer the question.\n\n"
        f"Question: {query}\n\n"
        f"Excerpt (page {page_node.page_range[0]}):\n{page_node.raw_text[:3000]}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": answer_prompt}],
        max_tokens=300,
        temperature=0,
    )
    answer = response.choices[0].message.content.strip()
    return answer

---

## 6. Sample Queries from the Annual Report

The questions below are representative of what students or analysts would ask from a BSE-listed company's annual report.

In [ ]:
# Query 1: Revenue / Financial performance
answer = vectorless_query(
    "What was the total revenue or turnover reported for the financial year?",
    root,
)
print("\nAnswer:")
print(answer)

In [ ]:
# Query 2: Board of Directors
answer = vectorless_query(
    "Who are the members of the Board of Directors mentioned in the report?",
    root,
)
print("\nAnswer:")
print(answer)

In [ ]:
# Query 3: Dividend declaration
answer = vectorless_query(
    "Was any dividend declared or recommended? If so, what is the amount per share?",
    root,
)
print("\nAnswer:")
print(answer)

In [ ]:
# Query 4: Auditor's remarks
answer = vectorless_query(
    "What are the key observations or qualifications made by the auditors?",
    root,
)
print("\nAnswer:")
print(answer)

---

## 7. Traditional RAG Baseline (Comparison)

For comparison, here is a minimal traditional RAG pipeline that chunks the same pages and retrieves by keyword overlap (TF-IDF) rather than vector embeddings.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def build_traditional_index(pages: List[str], chunk_size: int = 300):
    """Chunk pages into fixed-size segments and build a TF-IDF index."""
    chunks = []
    for text in pages:
        words = text.split()
        for i in range(0, max(1, len(words)), chunk_size):
            chunk = " ".join(words[i : i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)
    vectoriser = TfidfVectorizer()
    matrix = vectoriser.fit_transform(chunks)
    return chunks, vectoriser, matrix


def traditional_query(query: str, chunks, vectoriser, matrix, top_k: int = 2) -> str:
    """Retrieve top-k chunks by TF-IDF similarity, then answer with the LLM."""
    q_vec = vectoriser.transform([query])
    sims = cosine_similarity(q_vec, matrix).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]
    context = "\n\n---\n\n".join(chunks[i] for i in top_indices)

    prompt = (
        f"Use the document excerpts below to answer the question.\n\n"
        f"Question: {query}\n\nExcerpts:\n{context}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


# Build and query
chunks, vectoriser, matrix = build_traditional_index(DEMO_PAGES)
print(f"Total chunks created: {len(chunks)}")

trad_answer = traditional_query(
    "What was the total revenue or turnover reported for the financial year?",
    chunks, vectoriser, matrix,
)
print("\nTraditional RAG Answer:")
print(trad_answer)

---

## 8. Side-by-Side Comparison

In [ ]:
query = "What was the total revenue or turnover reported for the financial year?"

v_answer = vectorless_query(query, root)
t_answer = traditional_query(query, chunks, vectoriser, matrix)

print("=" * 60)
print("QUERY:", query)
print("=" * 60)
print("\n[Vectorless RAG]")
print(v_answer)
print("\n[Traditional RAG]")
print(t_answer)

---

## 9. Summary and Key Takeaways

| | Vectorless RAG | Traditional RAG |
|---|---|---|
| Index type | Hierarchical tree with summaries | Flat list of TF-IDF / dense vectors |
| Retrieval logic | LLM-guided traversal | Similarity search |
| Good at | Structured sections, tables, reports | Broad corpora, open-domain Q&A |
| Limitation | Does not scale to millions of docs | Loses document structure |

**Hybrid RAG** is the recommended production approach:
- Use the hierarchical index for **structured reasoning** (section identification).
- Fall back to vector search for **broad coverage** when the index traversal yields low confidence.

---

## 10. Exercises

1. Increase `section_size` to 10 pages. Does retrieval accuracy change for the dividend query?
2. Replace the TF-IDF baseline with dense embeddings (`text-embedding-3-small`). Compare latency.
3. Add a **confidence score** to `select_best_child` so the system can fall back to vector search when uncertain.
4. Extend the tree to three levels: document → chapter → section → page.